# runbook_diagnostics — Pipeline Diagnostic Runbook

> **Execute este notebook quando o pipeline falhar ou produzir resultados inesperados.**  
> Percorra as seções **em ordem**. Pare no primeiro `[FAIL]` e siga as etapas de remediação.  
> Corrija o problema, re-execute o passo afetado e re-execute esta seção para confirmar.

## Cobertura

| Seção | Domínio | Steps cobertos |
|---|---|---|
| 0 | Ambiente e utils | Todos os notebooks |
| 1 | Bootstrap log | Scripts 01–04 |
| 2 | Landing zone | Scripts 01–04 |
| 3 | Landing gate | 05_landing_validate |
| 4 | Conectividade de APIs | Scripts 01–04 |
| 5 | Camada Bronze | Notebooks 06–12 |
| 6 | Camada Silver | Notebooks 13–15 |
| 7 | Camada Gold | Notebooks 16–20 |
| 8 | Camada Serving | Notebooks 21–23 |
| 9 | Problemas conhecidos DuckDB/utils | Todos |
| 10 | Lookup de códigos de erro | Todos |

## Como usar
1. Execute **Setup** primeiro — sempre
2. Execute seções em ordem até encontrar um `[FAIL]`
3. Siga as etapas de remediação impressas abaixo do `[FAIL]`
4. Corrija, re-execute o step afetado, depois re-execute esta seção

## Changelog
| Data | Mudança |
|---|---|
| 2026-04-28 | Versão inicial — cobertura até 05_landing_validate |
| 2026-05-07 | Estendido para Bronze (06–12) e Silver (13–15); Seções 5–8; E40–E78 |
| 2026-05-08 | Versão final: Seções 7–8 Gold/Serving; E79–E95; erros da sessão 08/05 |

---
## Setup — imports e configuração

> **Sempre execute esta célula primeiro.**

In [ ]:
import json
import sys
import zipfile
import os
import requests
from datetime import datetime, timezone
from pathlib import Path

# --- Resolve root (funciona de qualquer profundidade de notebook) ------------
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent.parent  # ops/ -> notebooks/ -> project/
except NameError:
    _root_candidate = Path.cwd()
    for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
               Path.cwd().parent.parent.parent]:
        if (_c / 'utils' / 'paths.py').exists():
            _root_candidate = _c
            break

sys.path.insert(0, str(_root_candidate / 'utils'))

from paths        import get_project_root, to_sql_path, gold_path, serving_path, silver_path
from duckdb_utils import get_connection
from dotenv       import load_dotenv

PROJECT_ROOT = get_project_root()
LOG_PATH     = PROJECT_ROOT / 'db' / 'bootstrap_log.json'
GATE_PATH    = PROJECT_ROOT / 'db' / 'landing_gate.json'
BRONZE_DIR   = PROJECT_ROOT / 'data' / 'bronze'
SILVER_DIR   = PROJECT_ROOT / 'data' / 'silver'
GOLD_DIR     = PROJECT_ROOT / 'data' / 'gold'
SERVING_DIR  = PROJECT_ROOT / 'data' / 'serving'
DB_FILE      = PROJECT_ROOT / 'data' / 'bronze.duckdb'

SOURCES = {
    'receita_federal': PROJECT_ROOT / 'data' / 'raw' / 'receita_federal',
    'pncp'           : PROJECT_ROOT / 'data' / 'raw' / 'pncp',
    'transparencia'  : PROJECT_ROOT / 'data' / 'raw' / 'portal_transparencia',
    'compras_gov'    : PROJECT_ROOT / 'data' / 'raw' / 'compras_gov',
}

load_dotenv(PROJECT_ROOT / '.env')

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Python       : {sys.version.split()[0]}')
print(f'Timestamp    : {datetime.now(timezone.utc).isoformat()}')
print()
print('Setup completo — prossiga com a Seção 0.')

---
## Seção 0 — Ambiente e utils

**Diagnostica:** ambiente quebrado, dependências faltando, falhas de import de utils, erros de resolução de PROJECT_ROOT.

**Execute quando:** qualquer notebook falha com `ImportError`, `ModuleNotFoundError`, `NameError` em variáveis do projeto, ou erros de path.

**Códigos de erro:** E00–E04

In [ ]:
print('=== Seção 0 — Ambiente e utils ===')
print()

# 0.1 Versão Python
major, minor = sys.version_info[:2]
ok = (major == 3 and minor >= 11)
print(f"  [{'OK  ' if ok else 'FAIL'}] 0.1 Python {major}.{minor}")
if not ok:
    print('         -> Instale Python 3.11+ e recrie o .venv')

# 0.2 Pacotes obrigatórios
REQUIRED_PKGS = [
    ('duckdb',     'duckdb'),
    ('requests',   'requests'),
    ('dotenv',     'python-dotenv'),
    ('dateutil',   'python-dateutil'),
    ('pandas',     'pandas'),
    ('pyarrow',    'pyarrow'),
    ('sklearn',    'scikit-learn'),
]
for pkg, pip_name in REQUIRED_PKGS:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  [OK  ] 0.2 {pkg} {ver}')
    except ImportError:
        print(f'  [FAIL] E01 -- pacote {pkg} NAO instalado')
        print(f'         -> pip install {pip_name}')

# 0.3 Módulos utils
UTILS_MODULES = [
    ('paths',         'get_project_root'),
    ('paths',         'gold_path'),
    ('paths',         'serving_path'),
    ('duckdb_utils',  'get_connection'),
    ('duckdb_utils',  'cnpj_normalize_expr'),
    ('duckdb_utils',  'safe_date_expr'),
    ('validation',    'CheckSuite'),
    ('privacy',       'CNPJTokeniser'),
    ('bootstrap_log', 'make_entry'),
    ('pipeline',      'check_landing_gate'),
]
for module, symbol in UTILS_MODULES:
    try:
        mod = __import__(module)
        ok  = hasattr(mod, symbol)
        print(f"  [{'OK  ' if ok else 'FAIL'}] 0.3 utils/{module}.py -> {symbol}")
        if not ok:
            print(f'         -> E02: {symbol} ausente em utils/{module}.py — verifique versão do arquivo')
    except ImportError as e:
        print(f'  [FAIL] E73 -- utils/{module}.py -- {e}')
        print('         -> Use resolução de 3 níveis no Setup do notebook')
        print('         -> Veja Seção 9 / E73 para o padrão correto')

# 0.4 Estrutura de diretórios
EXPECTED_DIRS = ['data', 'data/bronze', 'data/silver', 'data/gold', 'data/serving',
                 'data/raw', 'utils', 'notebooks', 'db']
for d in EXPECTED_DIRS:
    exists = (PROJECT_ROOT / d).exists()
    print(f"  [{'OK  ' if exists else 'FAIL'}] 0.4 {d}/")
    if not exists:
        print(f'         -> Diretório {d}/ ausente')
        print(f'         -> PROJECT_ROOT resolvido para: {PROJECT_ROOT}')
        print('         -> Verifique resolução de root — veja E73')

# 0.5 Arquivo .env
env_path = PROJECT_ROOT / '.env'
env_ok = env_path.exists()
print(f"  [{'OK  ' if env_ok else 'FAIL'}] 0.5 .env em {env_path}")
if not env_ok:
    print('         -> E03: Crie .env a partir de .env.example')
    print('         -> Obrigatório: CNPJ_SALT, TRANSPARENCIA_API_KEY, RECEITA_FEDERAL_SHARE_TOKEN')

# 0.6 Variáveis de ambiente obrigatórias
ENV_VARS = [
    ('CNPJ_SALT',                    'Pseudonimização LGPD — camada Gold/Serving'),
    ('TRANSPARENCIA_API_KEY',        'Bootstrap Portal da Transparência'),
    ('RECEITA_FEDERAL_SHARE_TOKEN',  'Bootstrap Receita Federal Nextcloud'),
]
for var, purpose in ENV_VARS:
    val = os.getenv(var)
    ok  = bool(val)
    print(f"  [{'OK  ' if ok else 'FAIL'}] 0.6 {var}: {'definida' if ok else 'NAO DEFINIDA'}  ({purpose})")
    if not ok:
        print(f'         -> E04: Adicione {var}=<valor> ao .env')

print()
print('  Seção 0 completa.')

---
## Seção 1 — Análise do bootstrap log

**Diagnostica:** downloads incompletos, períodos ERROR/EMPTY, log ausente, fontes nunca inicializadas.

**Execute quando:** `05_landing_validate` falha ou Bronze não encontra arquivos raw.

**Códigos de erro:** E10–E14

In [ ]:
print('=== Seção 1 — Análise do bootstrap log ===')
print()

if not LOG_PATH.exists():
    print(f'  [FAIL] E10 -- bootstrap_log.json NAO ENCONTRADO: {LOG_PATH}')
    print('         -> Execute os scripts 01-04 em ordem antes de qualquer outro passo:')
    print('         -> 01_bootstrap_receita_federal.py')
    print('         -> 02_bootstrap_pncp.py')
    print('         -> 03_bootstrap_transparencia.py')
    print('         -> 04_bootstrap_compras.py')
else:
    con1 = get_connection()
    log_sql = to_sql_path(LOG_PATH)
    con1.execute(f"CREATE OR REPLACE TABLE _log AS SELECT * FROM read_json_auto('{log_sql}')")
    print('  [OK  ] 1.1 bootstrap_log.json encontrado')

    expected = {'receita_federal', 'pncp', 'portal_transparencia', 'compras_gov'}
    actual   = set(r[0] for r in con1.execute('SELECT DISTINCT source_id FROM _log').fetchall())
    missing  = expected - actual
    if missing:
        print(f'  [FAIL] E11 -- fontes nunca inicializadas: {missing}')
        script_map = {'receita_federal':'01','pncp':'02','portal_transparencia':'03','compras_gov':'04'}
        for s in missing:
            print(f'         -> Execute {script_map.get(s,"??")}_bootstrap_*.py')
    else:
        print('  [OK  ] 1.2 Todas as 4 fontes têm entradas no log')

    errors = con1.execute("""
        SELECT source_id, period, error_message FROM _log
        WHERE status = 'ERROR' ORDER BY source_id, period
    """).df()
    if len(errors):
        print(f'  [FAIL] E12 -- {len(errors)} período(s) com ERROR:')
        print(errors.to_string(index=False))
        print('         -> Re-execute o bootstrap para cada source/period com falha')
    else:
        print('  [OK  ] 1.3 Nenhum período com ERROR')

    empties = con1.execute("SELECT source_id, period FROM _log WHERE status = 'EMPTY'").df()
    if len(empties):
        print(f'  [WARN] E13 -- {len(empties)} período(s) EMPTY:')
        print(empties.to_string(index=False))
        print('         -> Verifique disponibilidade da API (Seção 4) e tente novamente')
    else:
        print('  [OK  ] 1.4 Nenhum período EMPTY')

    print()
    print('  Última execução bem-sucedida por fonte:')
    now = datetime.now(timezone.utc)
    for row in con1.execute("""
        SELECT source_id, MAX(finished_at) AS last_run
        FROM _log WHERE status IN ('SUCCESS','SKIPPED')
        GROUP BY source_id ORDER BY source_id
    """).fetchall():
        source_id, last_run = row
        try:
            last_dt  = datetime.fromisoformat(last_run.replace('Z', '+00:00'))
            days_ago = (now - last_dt).days
            flag     = 'WARN E14 -- atrasado' if days_ago > 35 else 'OK'
            print(f'    {source_id:<25} {last_run[:10]} ({days_ago}d atrás)  [{flag}]')
            if days_ago > 35:
                print(f'         -> Execute atualização mensal para {source_id}')
        except Exception:
            print(f'    {source_id:<25} {last_run}')

    print()
    summary = con1.execute("""
        SELECT source_id,
            COUNT(*) FILTER (WHERE status='SUCCESS') AS success,
            COUNT(*) FILTER (WHERE status='ERROR')   AS errors,
            COUNT(*) FILTER (WHERE status='EMPTY')   AS empty,
            SUM(records)                             AS total_records,
            ROUND(SUM(bytes_written)/1048576.0,1)    AS total_mb
        FROM _log GROUP BY source_id ORDER BY source_id
    """).df()
    print('  Resumo de volume:')
    print(summary.to_string(index=False))
    con1.close()

print()
print('  Seção 1 completa.')

---
## Seção 2 — Inventário da landing zone

**Diagnostica:** diretórios de fonte ausentes, arquivos vazios, estrutura errada, ZIPs corrompidos.

**Execute quando:** `05_landing_validate` reporta MISSING ou WARN para qualquer fonte.

**Códigos de erro:** E20–E24

In [ ]:
print('=== Seção 2 — Inventário da landing zone ===')
print()

for source_id, src_path in SOURCES.items():
    if not src_path.exists():
        print(f'  [FAIL] E20 -- {source_id}/ ausente: {src_path}')
        print(f'         -> Execute o bootstrap para {source_id}')
        continue

    all_files = list(src_path.rglob('*'))
    files     = [f for f in all_files if f.is_file()]
    total_mb  = sum(f.stat().st_size for f in files) / 1_048_576
    print(f'  [OK  ] {source_id:<20} {len(files):>4} arquivo(s)  {total_mb:>8.1f} MB')

    # Arquivos vazios ou muito pequenos
    tiny = [f for f in files if f.stat().st_size < 100]
    if tiny:
        print(f'  [WARN] E21 -- {len(tiny)} arquivo(s) suspeito(s) (< 100 bytes):')
        for f in tiny[:5]:
            print(f'           {f.name} ({f.stat().st_size} bytes)')
        print('         -> Delete e re-execute o bootstrap para esses períodos')

    # Receita Federal: espera 27 ZIPs
    if source_id == 'receita_federal':
        zips = [f for f in files if f.suffix == '.zip']
        ok   = len(zips) == 27
        print(f"  [{'OK  ' if ok else 'FAIL'}] E22 -- RF ZIPs: {len(zips)} (esperado: 27)")
        if not ok:
            print('         -> Re-execute 01_bootstrap_receita_federal.py')

        # Verifica integridade dos ZIPs
        corrupt = []
        for z in zips:
            try:
                with zipfile.ZipFile(z) as zf:
                    zf.testzip()
            except Exception:
                corrupt.append(z.name)
        if corrupt:
            print(f'  [FAIL] E23 -- {len(corrupt)} ZIP(s) corrompido(s): {corrupt}')
            print('         -> Delete os ZIPs corrompidos e re-execute 01_bootstrap_receita_federal.py')
        else:
            print(f'  [OK  ] E23 -- Todos os {len(zips)} ZIPs íntegros')

    # Compras.gov: verifica estrutura de subdiretórios
    if source_id == 'compras_gov':
        subdirs = [d for d in src_path.iterdir() if d.is_dir()]
        json_files = list(src_path.rglob('*.json'))
        old_struct = any('_' not in d.name for d in subdirs)
        if old_struct and not json_files:
            print('  [WARN] E24 -- estrutura antiga de compras_gov detectada')
            print('         -> Delete compras_gov/ e re-execute 04_bootstrap_compras.py')

print()
print('  Seção 2 completa.')

---
## Seção 3 — Landing gate

**Diagnostica:** gate ausente, status diferente de SUCCESS, gate desatualizado em relação ao bootstrap.

**Execute quando:** qualquer notebook Bronze ou Silver falha na verificação do gate.

**Códigos de erro:** E30–E32

In [ ]:
print('=== Seção 3 — Landing gate ===')
print()

if not GATE_PATH.exists():
    print(f'  [FAIL] E30 -- landing_gate.json NAO ENCONTRADO: {GATE_PATH}')
    print('         -> Execute 05_landing_validate.ipynb')
else:
    with open(GATE_PATH) as f:
        gate = json.load(f)

    status = gate.get('status', 'UNKNOWN')
    ok     = status == 'SUCCESS'
    print(f"  [{'OK  ' if ok else 'FAIL'}] E31 -- gate status: {status}")
    if not ok:
        print('         -> Corrija os problemas das Seções 1-2')
        print('         -> Re-execute 05_landing_validate.ipynb')

    gate_ts = gate.get('validated_at', '')
    if gate_ts and LOG_PATH.exists():
        con3 = get_connection()
        log_sql = to_sql_path(LOG_PATH)
        con3.execute(f"CREATE OR REPLACE TABLE _log3 AS SELECT * FROM read_json_auto('{log_sql}')")
        last_boot = con3.execute("""
            SELECT MAX(finished_at) FROM _log3
            WHERE status IN ('SUCCESS','SKIPPED')
        """).fetchone()[0]
        con3.close()
        if last_boot and last_boot > gate_ts:
            print(f'  [WARN] E32 -- gate ({gate_ts[:10]}) desatualizado em relação ao bootstrap ({last_boot[:10]})')
            print('         -> Re-execute 05_landing_validate.ipynb após o último bootstrap')
        else:
            print(f'  [OK  ] E32 -- gate atualizado ({gate_ts[:10]})')

    print()
    print('  Detalhes do gate:')
    for k, v in gate.items():
        print(f'    {k}: {v}')

print()
print('  Seção 3 completa.')

---
## Seção 4 — Conectividade de APIs

**Diagnostica:** APIs indisponíveis, tokens expirados, endpoints renomeados.

**Execute quando:** bootstrap falha com erros HTTP 4xx/5xx ou timeout.

**Códigos de erro:** E40–E44

In [ ]:
print('=== Seção 4 — Conectividade de APIs ===')
print()

TRANSPARENCIA_KEY = os.getenv('TRANSPARENCIA_API_KEY', '')

API_CHECKS = [
    (
        'Compras.gov',
        'https://contratos.comprasnet.gov.br/api/contrato?codigoOrgao=36000&dataInicio=2024-01-01&dataFim=2024-01-31&tamanhoPagina=10&pagina=0',
        {},
        'E40/E41',
        'Erro HTTP ou JPA crash — aguarde 30-60 min (problema server-side). tamanhoPagina deve estar entre 10-500.'
    ),
    (
        'PNCP',
        'https://pncp.gov.br/api/consulta/v1/contratos?dataInicial=2024-01-01&dataFinal=2024-01-31&pagina=1&tamanhoPagina=10',
        {},
        'E42',
        'Timeout ou 404 — tente novamente mais tarde. Endpoint correto: /api/consulta/v1/contratos'
    ),
    (
        'Portal Transparencia',
        'https://api.portaldatransparencia.gov.br/api-de-dados/ceis?pagina=1&tamanhoPagina=5',
        {'chave-api': TRANSPARENCIA_KEY} if TRANSPARENCIA_KEY else {},
        'E43',
        'HTTP 401/403 ou chave ausente — renove em portaldatransparencia.gov.br'
    ),
]

for name, url, headers, code, remediation in API_CHECKS:
    try:
        r = requests.get(url, headers=headers, timeout=15)
        ok = r.status_code in (200, 206)
        print(f"  [{'OK  ' if ok else 'FAIL'}] {name:<25} HTTP {r.status_code}")
        if not ok:
            print(f'         -> {code}: {remediation}')
    except requests.exceptions.Timeout:
        print(f'  [FAIL] {code} -- {name}: TIMEOUT')
        print(f'         -> {remediation}')
    except requests.exceptions.ConnectionError as e:
        print(f'  [FAIL] {code} -- {name}: ConnectionError -- {str(e)[:60]}')
        print(f'         -> Verifique conectividade de rede')

# Receita Federal (Nextcloud share)
RF_TOKEN = os.getenv('RECEITA_FEDERAL_SHARE_TOKEN', '')
if RF_TOKEN:
    try:
        rf_url = f'https://dadosabertos.rfb.gov.br/CNPJ/dados_abertos_cnpj/'
        r = requests.get(rf_url, timeout=10)
        ok = r.status_code == 200
        print(f"  [{'OK  ' if ok else 'FAIL'}] {'Receita Federal':<25} HTTP {r.status_code}")
        if not ok:
            print(f'         -> E44: Token rotacionado ou mês ainda não publicado (dias 8-12)')
    except Exception as e:
        print(f'  [WARN] Receita Federal -- {str(e)[:60]}')
else:
    print('  [WARN] E44 -- RECEITA_FEDERAL_SHARE_TOKEN nao definida — skip verificacao RF')

print()
print('  Seção 4 completa.')

---
## Seção 5 — Camada Bronze

**Diagnostica:** arquivos Bronze ausentes, contagem de linhas abaixo do mínimo, bronze.duckdb corrompido.

**Execute quando:** notebooks Silver falham ao ler Bronze, ou após re-run de qualquer notebook Bronze.

**Códigos de erro:** E50–E56

In [ ]:
print('=== Seção 5 — Camada Bronze ===')
print()

BRONZE_SOURCES = {
    'rf_empresas':         {'min_rows': 50_000_000,  'notebook': '09_bronze_rf_empresas'},
    'rf_estabelecimentos': {'min_rows': 50_000_000,  'notebook': '09_bronze_rf_estabelecimentos'},
    'rf_simples':          {'min_rows': 30_000_000,  'notebook': '10_bronze_rf_simples'},
    'rf_dominios':         {'min_rows': 500,         'notebook': '11_bronze_rf_dominios'},
    'rf_paises':           {'min_rows': 200,         'notebook': '11_bronze_rf_dominios'},
    'rf_qualificacoes':    {'min_rows': 50,          'notebook': '11_bronze_rf_dominios'},
    'rf_motivos':          {'min_rows': 50,          'notebook': '11_bronze_rf_dominios'},
    'compras_contratos':   {'min_rows': 500_000,     'notebook': '06_bronze_compras_contratos'},
    'pncp_contratos':      {'min_rows': 100_000,     'notebook': '07_bronze_pncp_contratos'},
    'transp_ceis':         {'min_rows': 10_000,      'notebook': '12_bronze_transp_ceis_cnep'},
    'transp_cnep':         {'min_rows': 1_000,       'notebook': '12_bronze_transp_ceis_cnep'},
}

if not BRONZE_DIR.exists():
    print(f'  [FAIL] E50 -- diretório Bronze NAO ENCONTRADO: {BRONZE_DIR}')
    print('         -> Execute os notebooks Bronze 06-12 em ordem')
else:
    con5 = get_connection()
    for src, cfg in BRONZE_SOURCES.items():
        src_dir = BRONZE_DIR / src
        if not src_dir.exists():
            print(f'  [FAIL] E50 -- {src}/ ausente -> Execute {cfg["notebook"]}.ipynb')
            continue
        files = list(src_dir.rglob('*.parquet'))
        if not files:
            print(f'  [FAIL] E51 -- {src}/ sem Parquet -> Execute {cfg["notebook"]}.ipynb')
            continue
        total_mb = sum(f.stat().st_size for f in files) / 1_048_576
        path_sql = to_sql_path(src_dir) + '/**/*.parquet'
        try:
            n  = con5.execute(f"SELECT count(*) FROM read_parquet('{path_sql}')").fetchone()[0]
            ok = n >= cfg['min_rows']
            print(f"  [{'OK  ' if ok else 'WARN'}] {src:<30} {n:>12,} rows  {total_mb:>8.1f} MB")
            if not ok:
                print(f'         -> E52 -- {n:,} rows < mínimo {cfg["min_rows"]:,}')
                print(f'         -> Re-execute {cfg["notebook"]}.ipynb')
        except Exception as e:
            print(f'  [FAIL] E53 -- {src}: {str(e)[:80]}')
            print(f'         -> Delete {src}/ e re-execute {cfg["notebook"]}.ipynb')
    con5.close()

# bronze.duckdb
print()
print('  --- bronze.duckdb ---')
if not DB_FILE.exists():
    print('  [INFO] E54 -- bronze.duckdb nao encontrado')
    print('         -> Normal antes de 15_silver_identidade rodar')
else:
    size_gb = DB_FILE.stat().st_size / (1024**3)
    print(f'  [OK  ] bronze.duckdb existe -- {size_gb:.1f} GB')
    wal_file = DB_FILE.parent / 'bronze.duckdb.wal'
    tmp_dir  = DB_FILE.parent / 'bronze.duckdb.tmp'
    if wal_file.exists():
        wal_kb = wal_file.stat().st_size / 1024
        print(f'  [WARN] E55 -- bronze.duckdb.wal existe ({wal_kb:.0f} KB)')
        print('         -> WAL de conexão não fechada. Delete se nenhum notebook estiver rodando.')
    if tmp_dir.exists():
        print('  [WARN] E55 -- bronze.duckdb.tmp existe')
        print('         -> Diretório de spill. Delete se nenhum notebook estiver rodando.')
    try:
        con_db = get_connection(db_path=DB_FILE)
        for table in ['rf_estabelecimentos', 'rf_empresas', 'rf_simples', 'cnpjs_ativos']:
            result = con_db.execute(f"""
                SELECT COUNT(*) FROM information_schema.tables WHERE table_name = '{table}'
            """).fetchone()[0]
            if result:
                n = con_db.execute(f'SELECT count(*) FROM {table}').fetchone()[0]
                print(f'  [OK  ] {table:<30} {n:>15,} rows')
            else:
                flag = 'INFO' if table == 'cnpjs_ativos' else 'WARN E56'
                print(f'  [{flag}] {table:<30} NAO encontrado em bronze.duckdb')
                if table != 'cnpjs_ativos':
                    print('         -> Delete bronze.duckdb e re-execute 15_silver_identidade a partir do Step 2')
        con_db.close()
    except Exception as e:
        print(f'  [FAIL] E56 -- Nao foi possivel abrir bronze.duckdb: {e}')

print()
print('  Seção 5 completa.')

---
## Seção 6 — Camada Silver

**Diagnostica:** arquivos Silver ausentes, contagens erradas, ADR-007 não aplicado, tipos de data incorretos, falhas de integridade do cnpj_normalized.

**Execute quando:** notebooks Gold falham ao ler Silver, ou após qualquer re-run de Silver.

**Códigos de erro:** E60–E65

In [ ]:
print('=== Seção 6 — Camada Silver ===')
print()

SILVER_SOURCES = {
    'silver_ceis':       {'min_rows': 10_000,  'partition': False, 'notebook': '13_silver_sancoes'},
    'silver_cnep':       {'min_rows': 1_000,   'partition': False, 'notebook': '13_silver_sancoes'},
    'silver_compras':    {'min_rows': 500_000, 'partition': True,  'notebook': '14_silver_contratual'},
    'silver_pncp':       {'min_rows': 100_000, 'partition': False, 'notebook': '14_silver_contratual'},
    'silver_identidade': {'min_rows': 500_000, 'partition': True,  'notebook': '15_silver_identidade'},
}

if not SILVER_DIR.exists():
    print(f'  [FAIL] E60 -- diretório Silver NAO ENCONTRADO: {SILVER_DIR}')
    print('         -> Execute os notebooks Silver 13, 14, 15 nessa ordem')
else:
    con6 = get_connection()
    silver_counts = {}

    for src, cfg in SILVER_SOURCES.items():
        src_dir = SILVER_DIR / src
        if not src_dir.exists():
            print(f'  [FAIL] E60 -- {src}/ ausente -> Execute {cfg["notebook"]}.ipynb')
            continue
        if cfg['partition']:
            files    = list(src_dir.rglob('*.parquet'))
            path_sql = to_sql_path(src_dir) + '/**/*.parquet'
        else:
            data_file = src_dir / 'data.parquet'
            files     = [data_file] if data_file.exists() else []
            path_sql  = to_sql_path(data_file)
        if not files:
            print(f'  [FAIL] E61 -- {src}/ sem Parquet -> Execute {cfg["notebook"]}.ipynb')
            continue
        total_mb = sum(f.stat().st_size for f in files) / 1_048_576
        try:
            n = con6.execute(f"SELECT count(*) FROM read_parquet('{path_sql}')").fetchone()[0]
            silver_counts[src] = n
            ok = n >= cfg['min_rows']
            print(f"  [{'OK  ' if ok else 'WARN'}] {src:<25} {n:>12,} rows  {total_mb:>8.1f} MB  {len(files)} arquivo(s)")
            if not ok:
                print(f'         -> E62 -- {n:,} rows < mínimo {cfg["min_rows"]:,}')
                print(f'         -> Re-execute {cfg["notebook"]}.ipynb')
        except Exception as e:
            print(f'  [FAIL] E63 -- {src}: {str(e)[:100]}')

    # ADR-007
    print()
    print('  --- Validação ADR-007 (silver_identidade) ---')
    id_count = silver_counts.get('silver_identidade', 0)
    if id_count == 0:
        print('  [SKIP] silver_identidade indisponível — execute 15_silver_identidade primeiro')
    elif id_count > 5_000_000:
        print(f'  [FAIL] E64 -- silver_identidade tem {id_count:,} rows — filtro ADR-007 NAO aplicado')
        print('         -> Esperado: ~810k rows (apenas CNPJs com atividade em contratos/sanções)')
        print('         -> Fix:')
        print('            1. Confirme que silver_ceis, silver_cnep, silver_compras, silver_pncp existem')
        print('            2. Delete data/silver/silver_identidade/')
        print('            3. Re-execute 15_silver_identidade a partir do Step 3 (reconstrói cnpjs_ativos)')
        print('            4. Execute Step 4 NA MESMA SESSÃO do kernel')
    else:
        print(f'  [OK  ] silver_identidade {id_count:,} rows — filtro ADR-007 aplicado')

    # Integridade cnpj_normalized
    print()
    print('  --- Integridade cnpj_normalized ---')
    for src in ['silver_ceis', 'silver_cnep', 'silver_compras', 'silver_pncp']:
        src_dir = SILVER_DIR / src
        if not src_dir.exists():
            continue
        data_file = src_dir / 'data.parquet'
        path_sql = to_sql_path(data_file) if data_file.exists() \
                   else to_sql_path(src_dir) + '/**/*.parquet'
        try:
            nulls = con6.execute(f"SELECT count(*) FROM read_parquet('{path_sql}') WHERE cnpj_normalized IS NULL").fetchone()[0]
            wrong = con6.execute(f"SELECT count(*) FROM read_parquet('{path_sql}') WHERE length(cnpj_normalized) != 14").fetchone()[0]
            ok = (nulls == 0 and wrong == 0)
            print(f"  [{'OK  ' if ok else 'FAIL'}] {src:<25} cnpj nulls={nulls}  wrong_len={wrong}")
            if not ok:
                print(f'         -> E65 -- falha de integridade cnpj_normalized')
                print("         -> ADR-002: deve ser VARCHAR 14 chars, padrão '[^0-9A-Za-z]'")
        except Exception as e:
            print(f'  [SKIP] {src}: {str(e)[:60]}')

    # Tipos de data silver_identidade
    print()
    print('  --- Tipos de data silver_identidade ---')
    id_dir = SILVER_DIR / 'silver_identidade'
    if id_dir.exists() and list(id_dir.rglob('*.parquet')):
        path_sql = to_sql_path(id_dir) + '/**/*.parquet'
        try:
            schema   = con6.execute(f"DESCRIBE SELECT * FROM read_parquet('{path_sql}')").fetchall()
            date_cols = {r[0]: r[1] for r in schema if 'data_' in r[0]}
            for col, dtype in date_cols.items():
                ok = dtype == 'DATE'
                print(f"  [{'OK  ' if ok else 'FAIL'}] {col:<35} {dtype}")
                if not ok:
                    print(f'         -> E70 -- esperado DATE, obtido {dtype}')
                    if dtype == 'BIGINT':
                        print('         -> Use CASE WHEN val=0 THEN NULL')
                        print('                ELSE TRY_STRPTIME(CAST(val AS VARCHAR),')
                        print("                     '%Y%m%d')::DATE END")
        except Exception as e:
            print(f'  [SKIP] Nao foi possivel ler schema silver_identidade: {str(e)[:80]}')

    con6.close()

print()
print('  Seção 6 completa.')

---
## Seção 7 — Camada Gold

**Diagnostica:** arquivos Gold ausentes, contagens de linhas incorretas, checks de qualidade com falha.

**Execute quando:** notebooks Serving falham ao ler Gold, ou após qualquer re-run de Gold.

**Códigos de erro:** E79–E84

In [ ]:
print('=== Seção 7 — Camada Gold ===')
print()

GOLD_TABLES = {
    'dim_tempo':       {'min_rows': 3_000,       'max_rows': 5_000,       'notebook': '16_dim_tempo'},
    'dim_fornecedor':  {'min_rows': 800_000,     'max_rows': 900_000,     'notebook': '17_dim_fornecedor'},
    'fato_contratos':  {'min_rows': 4_000_000,   'max_rows': 6_000_000,   'notebook': '18_fato_contratos'},
    'fato_sancoes':    {'min_rows': 10_000,      'max_rows': 50_000,      'notebook': '19_fato_sancoes'},
}

if not GOLD_DIR.exists():
    print(f'  [FAIL] E79 -- diretório Gold NAO ENCONTRADO: {GOLD_DIR}')
    print('         -> Execute os notebooks Gold em ordem')
else:
    con7 = get_connection()

    for table, cfg in GOLD_TABLES.items():
        parquet_path = GOLD_DIR / f'{table}.parquet'
        if not parquet_path.exists():
            print(f'  [FAIL] E79 -- {table}.parquet ausente -> Execute {cfg["notebook"]}.ipynb')
            continue
        size_mb = parquet_path.stat().st_size / 1_048_576
        try:
            path_sql = to_sql_path(parquet_path)
            n = con7.execute(f"SELECT count(*) FROM read_parquet('{path_sql}')").fetchone()[0]
            ok = cfg['min_rows'] <= n <= cfg['max_rows']
            print(f"  [{'OK  ' if ok else 'WARN'}] {table:<20} {n:>12,} rows  {size_mb:>8.1f} MB")
            if not ok:
                print(f'         -> E80 -- {n:,} rows fora do intervalo esperado [{cfg["min_rows"]:,} – {cfg["max_rows"]:,}]')
                print(f'         -> Re-execute {cfg["notebook"]}.ipynb')
        except Exception as e:
            print(f'  [FAIL] E81 -- {table}: {str(e)[:80]}')

    # dim_fornecedor: SCD2 integrity checks
    print()
    print('  --- dim_fornecedor SCD2 integrity ---')
    dim_path = GOLD_DIR / 'dim_fornecedor.parquet'
    if dim_path.exists():
        path_sql = to_sql_path(dim_path)
        try:
            # Deve ter exatamente um is_current=true por supplier_sk
            dupes = con7.execute(f"""
                SELECT COUNT(*) FROM (
                    SELECT supplier_sk, COUNT(*) AS cnt
                    FROM read_parquet('{path_sql}')
                    WHERE is_current = true
                    GROUP BY supplier_sk HAVING cnt > 1
                )
            """).fetchone()[0]
            print(f"  [{'OK  ' if dupes == 0 else 'FAIL'}] E82 -- SCD2 duplicatas is_current=true: {dupes}")
            if dupes > 0:
                print('         -> E82: dim_fornecedor corrompida — delete e re-execute 17_dim_fornecedor.ipynb')

            # cnpj_normalized nao deve estar exposto no Gold
            cols = [r[0] for r in con7.execute(f"DESCRIBE SELECT * FROM read_parquet('{path_sql}')").fetchall()]
            has_cnpj = 'cnpj_normalized' in cols
            # cnpj_normalized PODE estar presente no Gold (dim_fornecedor usa para joins internos)
            # mas NAO deve estar exposto na camada Serving
            print(f'  [INFO] cnpj_normalized em dim_fornecedor: {has_cnpj} (ok para Gold, proibido em Serving)')
        except Exception as e:
            print(f'  [SKIP] Nao foi possivel verificar dim_fornecedor: {str(e)[:80]}')

    # fato_contratos: grain check (contrato_sk deve ser único)
    print()
    print('  --- fato_contratos grain check ---')
    fc_path = GOLD_DIR / 'fato_contratos.parquet'
    if fc_path.exists():
        path_sql = to_sql_path(fc_path)
        try:
            total = con7.execute(f"SELECT count(*) FROM read_parquet('{path_sql}')").fetchone()[0]
            distinct = con7.execute(f"SELECT count(DISTINCT contrato_sk) FROM read_parquet('{path_sql}')").fetchone()[0]
            ok = total == distinct
            print(f"  [{'OK  ' if ok else 'FAIL'}] E83 -- contrato_sk único: total={total:,} distinct={distinct:,}")
            if not ok:
                print('         -> E83: grain de fato_contratos violado — deve incluir codigo_unidade_gestora')
                print('         -> Veja ADR sobre grain de contratos')

            # Valores negativos
            neg = con7.execute(f"SELECT count(*) FROM read_parquet('{path_sql}') WHERE _valor_negativo = true").fetchone()[0]
            pct = neg / total * 100 if total > 0 else 0
            flag = 'WARN E84' if pct > 5 else 'OK'
            print(f'  [{flag}] E84 -- valores negativos: {neg:,} ({pct:.1f}%)')
            if pct > 5:
                print('         -> Percentual acima de 5% — verifique source Silver')
        except Exception as e:
            print(f'  [SKIP] Nao foi possivel verificar fato_contratos: {str(e)[:80]}')

    con7.close()

print()
print('  Seção 7 completa.')

---
## Seção 8 — Camada Serving

**Diagnostica:** arquivos Serving ausentes, leakage de CNPJ, target com distribuição errada, arquivos H2O com colunas incorretas.

**Execute quando:** notebooks 21-23 falham, ou antes de carregar dataset no H2O.

**Códigos de erro:** E85–E95

In [ ]:
print('=== Seção 8 — Camada Serving ===')
print()

SERVING_FILES = {
    'serving_fornecedor_perfil.parquet':         {'min_rows': 800_000, 'notebook': '20_serving_fornecedor_perfil'},
    'serving_fornecedor_features.parquet':       {'min_rows': 800_000, 'notebook': '21_serving_fornecedor_features'},
    'serving_fornecedor_features.csv':           {'min_rows': 800_000, 'notebook': '21_serving_fornecedor_features'},
    'serving_fornecedor_features_v2.parquet':    {'min_rows': 800_000, 'notebook': '23_feature_engineering_v3'},
    'serving_fornecedor_features_v3.parquet':    {'min_rows': 800_000, 'notebook': '23_feature_engineering_v3'},
    'h2o_supplier_risk_dataset.csv':             {'min_rows': 800_000, 'notebook': '22_h2o_dataset_preparation'},
    'h2o_supplier_risk_dataset_v3.csv':          {'min_rows': 800_000, 'notebook': '23_feature_engineering_v3'},
    'h2o_supplier_risk_dataset_train.csv':       {'min_rows': 600_000, 'notebook': '22_h2o_dataset_preparation'},
    'h2o_supplier_risk_dataset_test.csv':        {'min_rows': 150_000, 'notebook': '22_h2o_dataset_preparation'},
    'h2o_supplier_risk_dataset_v3_train.csv':    {'min_rows': 600_000, 'notebook': '23_feature_engineering_v3'},
    'h2o_supplier_risk_dataset_v3_test.csv':     {'min_rows': 150_000, 'notebook': '23_feature_engineering_v3'},
}

if not SERVING_DIR.exists():
    print(f'  [FAIL] E85 -- diretório Serving NAO ENCONTRADO: {SERVING_DIR}')
else:
    for fname, cfg in SERVING_FILES.items():
        fpath = SERVING_DIR / fname
        if not fpath.exists():
            print(f'  [INFO] {fname} ausente -> Execute {cfg["notebook"]}.ipynb')
        else:
            size_mb = fpath.stat().st_size / 1_048_576
            print(f'  [OK  ] {fname:<50} {size_mb:>8.1f} MB')

# Verificações específicas do dataset v3
print()
print('  --- serving_fornecedor_features_v3: validação detalhada ---')
v3_path = SERVING_DIR / 'serving_fornecedor_features_v3.parquet'
if v3_path.exists():
    con8 = get_connection()
    path_sql = to_sql_path(v3_path)
    try:
        total = con8.execute(f"SELECT count(*) FROM read_parquet('{path_sql}')").fetchone()[0]
        ok = total == 810_921
        print(f"  [{'OK  ' if ok else 'FAIL'}] E86 -- row count: {total:,} (esperado: 810.921)")

        # Leakage check
        cols = [r[0] for r in con8.execute(f"DESCRIBE SELECT * FROM read_parquet('{path_sql}')").fetchall()]
        leakage = [c for c in ['qtd_sancoes','tem_ceis','tem_cnep','sancao_ativa',
                               'valor_total_multa','cnpj_normalized'] if c in cols]
        ok_leak = len(leakage) == 0
        print(f"  [{'OK  ' if ok_leak else 'FAIL'}] E87 -- leakage/LGPD check: {leakage if leakage else 'clean'}")
        if not ok_leak:
            print('         -> E87: Colunas de leakage ou CNPJ exposto detectados')
            print('         -> cnpj_normalized viola ADR-005; qtd_sancoes/tem_ceis/etc violam integridade do experimento')
            print('         -> Re-execute notebook 23 Step 15')

        # Target distribution
        pos = con8.execute(f"SELECT count(*) FROM read_parquet('{path_sql}') WHERE tem_sancao = true").fetchone()[0]
        neg = con8.execute(f"SELECT count(*) FROM read_parquet('{path_sql}') WHERE tem_sancao = false").fetchone()[0]
        ok_pos = pos == 9_946
        ok_neg = neg == 800_975
        print(f"  [{'OK  ' if ok_pos else 'FAIL'}] E88 -- positivos (tem_sancao=1): {pos:,} (esperado: 9.946)")
        print(f"  [{'OK  ' if ok_neg else 'FAIL'}] E88 -- negativos (tem_sancao=0): {neg:,} (esperado: 800.975)")

        # Column count
        ncols = len(cols)
        ok_cols = ncols == 71
        print(f"  [{'OK  ' if ok_cols else 'WARN'}] E89 -- colunas: {ncols} (esperado: 71)")
        if not ok_cols:
            print(f'         -> Diferença de colunas pode indicar feature group ausente')
            print(f'         -> Re-execute notebook 23 Steps 4-15')

    except Exception as e:
        print(f'  [FAIL] E85 -- Nao foi possivel ler serving_fornecedor_features_v3: {str(e)[:80]}')
    finally:
        con8.close()
else:
    print('  [INFO] serving_fornecedor_features_v3.parquet nao encontrado — execute notebook 23')

# CSV H2O v3: verifica quoting
print()
print('  --- h2o_supplier_risk_dataset_v3.csv: integridade do quoting ---')
csv_path = SERVING_DIR / 'h2o_supplier_risk_dataset_v3.csv'
if csv_path.exists():
    import csv as csv_module
    errors = 0
    try:
        with open(csv_path, newline='', encoding='utf-8') as f:
            reader = csv_module.reader(f)
            header = next(reader)
            expected_cols = len(header)
            for i, row in enumerate(reader, 1):
                if len(row) != expected_cols:
                    errors += 1
                    if errors <= 3:
                        print(f'  [FAIL] E90 -- linha {i}: esperado {expected_cols} colunas, obtido {len(row)}')
        ok = errors == 0
        print(f"  [{'OK  ' if ok else 'FAIL'}] E90 -- CSV quoting: {i:,} linhas verificadas, {errors} erros")
        if not ok:
            print('         -> E90: CSV com quoting incorreto — campos com vírgula interna nao estão entre aspas')
            print('         -> Fix: adicione QUOTE \'"\' na cláusula COPY do DuckDB')
            print('         -> Re-execute Step 17 do notebook 23')
    except Exception as e:
        print(f'  [FAIL] E90 -- Erro ao verificar CSV: {str(e)[:80]}')
else:
    print('  [INFO] h2o_supplier_risk_dataset_v3.csv nao encontrado — execute notebook 23 Step 17')

print()
print('  Seção 8 completa.')

---
## Seção 9 — Problemas conhecidos DuckDB/utils

**Esta seção é um guia de referência, não uma verificação automatizada.**

Encontre sua mensagem de erro abaixo e siga o diagnóstico e a correção.

**Códigos de erro:** E70–E78, E91–E95

In [ ]:
print('=== Seção 9 — Problemas conhecidos DuckDB/utils ===')
print()

KNOWN_ISSUES = [
    (
        'E70',
        "Binder Error: No function matches try_strptime(BIGINT, STRING_LIT)",
        [
            "Bronze Parquet armazena algumas colunas de data como BIGINT.",
            "TRY_STRPTIME aceita apenas VARCHAR. safe_date_expr() gera padrões VARCHAR.",
            "Ocorre em: 15_silver_identidade Step 4.",
        ],
        [
            "Use CASE WHEN hardcoded ao invés de safe_date_expr():",
            "  CASE WHEN val = 0 THEN NULL",
            "       ELSE TRY_STRPTIME(CAST(val AS VARCHAR), '%Y%m%d')::DATE",
            "  END",
            "Colunas BIGINT confirmadas: data_situacao_cadastral, data_inicio_atividade,",
            "  data_situacao_especial (rf_estabelecimentos); data_opcao_simples (rf_simples).",
        ]
    ),
    (
        'E71',
        "IOException: No files found .../data.parquet/data.parquet (path duplicado)",
        [
            "silver_path() de paths.py já inclui /data.parquet.",
            "Concatenar '/data.parquet' novamente duplica o segmento.",
        ],
        [
            "Use a variável de path diretamente:",
            "  ERRADO : f\"SELECT * FROM read_parquet('{SD}/data.parquet')\"",
            "  CORRETO: f\"SELECT * FROM read_parquet('{SD}')\"",
        ]
    ),
    (
        'E72',
        "TypeError: can't subtract offset-naive and offset-aware datetimes",
        [
            "STARTED_AT criado com datetime.now(timezone.utc) (offset-aware)",
            "mas elapsed calculado com datetime.now() (offset-naive).",
        ],
        [
            "Sempre use timezone.utc nos dois lados da subtração:",
            "  elapsed = (datetime.now(timezone.utc) - STARTED_AT).total_seconds()",
        ]
    ),
    (
        'E73',
        "ModuleNotFoundError: No module named 'paths'",
        [
            "sys.path.insert usa Path.cwd().parent que resolve para notebooks/",
            "quando o notebook está em notebooks/eda/ ou notebooks/ops/ (dois níveis).",
            "utils/ está na raiz do projeto, não dentro de notebooks/.",
        ],
        [
            "Use resolução de 3 níveis com fallback baseado em marcador:",
            "  try:",
            "      _root = Path(__vsc_ipynb_file__).resolve().parent.parent.parent",
            "  except NameError:",
            "      _root = Path.cwd()",
            "      for c in [Path.cwd(), Path.cwd().parent,",
            "                Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:",
            "          if (c / 'utils' / 'paths.py').exists():",
            "              _root = c; break",
            "  sys.path.insert(0, str(_root / 'utils'))",
        ]
    ),
    (
        'E74',
        "ADR-007 nao aplicado: silver_identidade tem 70M+ rows ao invés de ~810k",
        [
            "Tabela cnpjs_ativos estava vazia ou Step 4 rodou antes do Step 3.",
            "Também ocorre quando partições antigas bloqueiam o reprocessamento.",
        ],
        [
            "1. Confirme que silver_ceis/cnep/compras/pncp existem (Seção 6)",
            "2. Delete data/silver/silver_identidade/ completamente",
            "3. Abra 15_silver_identidade — execute Step 3 primeiro (constrói cnpjs_ativos)",
            "4. Execute Step 4 NA MESMA SESSÃO do kernel (cnpjs_ativos deve estar em escopo)",
            "5. Confirme ~810k rows na Seção 6 deste runbook",
        ]
    ),
    (
        'E75',
        "BinderException: Could not find key 'esfera' in struct (PNCP)",
        [
            "API PNCP renomeou orgaoEntidade.esfera para orgaoEntidade.esferaId.",
        ],
        [
            "Substitua orgaoEntidade.esfera por orgaoEntidade.esferaId em todas as queries.",
            "Bronze já armazena como coluna flat orgaoEntidade_esferaId.",
        ]
    ),
    (
        'E76',
        "Validation FAIL: row count mismatch após crescimento da fonte",
        [
            "EXPECTED_CEIS / EXPECTED_CNEP hardcoded com valores desatualizados.",
            "Fonte cresceu entre runs.",
        ],
        [
            "Nunca hardcode contagens esperadas. Derive dinamicamente:",
            "  EXPECTED = scalar(con, f\"SELECT count(*) FROM read_parquet('{BD}')",
            "                          WHERE pessoa_tipo IN (...)\") ",
        ]
    ),
    (
        'E77',
        "FileNotFoundError: [WinError 2] bronze.duckdb stat() antes do flush",
        [
            "DB_FILE.stat().st_size chamado logo após CREATE TABLE.",
            "No Windows, DuckDB pode não ter feito flush ainda.",
        ],
        [
            "Proteja stat() com verificação de existência:",
            "  size_str = f\"{DB_FILE.stat().st_size/(1024**3):.1f}GB\" \\",
            "             if DB_FILE.exists() else \"criando...\"",
        ]
    ),
    (
        'E78',
        "SyntaxError: f-string: expecting ':' or '}'",
        [
            "Python não permite modificador !r com expressão condicional dentro de f-string.",
        ],
        [
            "Extraia o valor para variável antes da f-string:",
            "  display_val = repr(val) if val else 'N/A'",
            "  print(f\"  sample={display_val}\")",
        ]
    ),
    (
        'E91',
        "ImportError: cannot import name 'GOLD_DIR' from 'utils.paths'",
        [
            "paths.py exporta funções, não constantes.",
            "Não existe GOLD_DIR — existe gold_path(ROOT, table_name).",
            "Ocorreu ao construir notebook 21 (serving_fornecedor_features).",
        ],
        [
            "Use o padrão de função:",
            "  from utils.paths import get_project_root, gold_path, serving_path",
            "  ROOT = get_project_root()",
            "  DIM_FORNECEDOR_PATH = gold_path(ROOT, 'dim_fornecedor')",
            "  OUTPUT_PATH = serving_path(ROOT, 'serving_fornecedor_features')",
        ]
    ),
    (
        'E92',
        "BinderException: Referenced column 'cnpj_token' not found (dim_fornecedor)",
        [
            "A chave exposta em dim_fornecedor é supplier_sk (MD5 hash), não cnpj_token.",
            "cnpj_token é o nome usado no serving_fornecedor_perfil.",
            "Ocorreu ao construir Step 3 do notebook 21.",
        ],
        [
            "Sempre verifique o schema real antes de escrever SQL:",
            "  con.execute('DESCRIBE dim_fornecedor').df()",
            "Use supplier_sk como chave em todas as tabelas Gold e Serving.",
        ]
    ),
    (
        'E93',
        "InvalidInputException: CSV Error — Expected N columns, Found N+M (quoting)",
        [
            "Campo categórico com vírgula interna (ex: 'Estabelecimento, no Brasil, de Sociedade Estrangeira')",
            "gravado sem quoting explícito no COPY do DuckDB.",
            "O parser do CSV interpreta as vírgulas internas como separadores de campo.",
            "Ocorreu ao ler h2o_supplier_risk_dataset.csv com read_csv_auto.",
        ],
        [
            "Na escrita: adicione QUOTE '\"' na cláusula COPY:",
            "  TO 'arquivo.csv' (FORMAT CSV, HEADER true, QUOTE '\"')",
            "Na leitura: especifique quote explicitamente:",
            "  read_csv('arquivo.csv', header=true, quote='\"', delim=',')",
            "Nunca use read_csv_auto para CSVs com campos categóricos complexos.",
        ]
    ),
    (
        'E94',
        "BinderException: Referenced column '_valor_negativo' not found in silver_pncp",
        [
            "_valor_negativo é coluna de fato_contratos (Gold), não de silver_pncp.",
            "silver_pncp usa valor_inicial diretamente — sem flag de negativo.",
            "Ocorreu ao construir feat_gC_cadastral_adv no notebook 23 Step 10.",
        ],
        [
            "Para silver_pncp, filtre valores diretamente por valor_inicial > 0:",
            "  SUM(CASE WHEN valor_inicial > 0 THEN valor_inicial ELSE 0 END)",
            "Reserve WHERE NOT _valor_negativo exclusivamente para fato_contratos.",
        ]
    ),
    (
        'E95',
        "ModuleNotFoundError: No module named 'sklearn'",
        [
            "scikit-learn nao está instalado no .venv.",
            "Necessário para train_test_split nos notebooks 22 e 23.",
        ],
        [
            "Instale no terminal do projeto (com .venv ativo):",
            "  pip install scikit-learn",
        ]
    ),
]

for code, error, cause_lines, fix_lines in KNOWN_ISSUES:
    print(f'  [{code}] {error}')
    print('  Causa:')
    for line in cause_lines:
        print(f'    {line}')
    print('  Fix:')
    for line in fix_lines:
        print(f'    {line}')
    print()

print('  Seção 9 completa.')

---
## Seção 10 — Lookup de códigos de erro

Tabela de referência rápida para todos os códigos de erro.

Defina `lookup = 'E91'` e re-execute para imprimir o detalhe completo de qualquer código.

In [ ]:
ERROR_CODES = {
    # Seção 0 — Ambiente
    'E00': ('utils/ nao está em sys.path',                           'Use resolução de 3 níveis — veja E73'),
    'E01': ('Pacote obrigatório nao instalado',                      'pip install <pacote> dentro do .venv'),
    'E02': ('Símbolo ausente em módulo utils',                       'Verifique versão do arquivo utils/'),
    'E03': ('Arquivo .env ausente',                                  'Crie .env a partir de .env.example'),
    'E04': ('Variável de ambiente obrigatória nao definida',         'Adicione var=<valor> ao .env'),
    # Seção 1 — Bootstrap log
    'E10': ('bootstrap_log.json ausente',                           'Execute todos os 4 scripts bootstrap 01-04'),
    'E11': ('Fonte nunca inicializada',                             'Execute script bootstrap para a fonte ausente'),
    'E12': ('Períodos ERROR no log',                                'Re-execute bootstrap para cada source/period com falha'),
    'E13': ('Períodos EMPTY — API não retornou dados',              'Verifique API (Seção 4) e tente novamente'),
    'E14': ('Bootstrap atrasado > 35 dias',                         'Execute atualização mensal do bootstrap'),
    # Seção 2 — Landing zone
    'E20': ('Diretório ou arquivo de fonte ausente',                'Execute bootstrap para essa fonte'),
    'E21': ('Arquivos vazios ou suspeitos (< 100 bytes)',           'Delete e re-execute bootstrap para esses períodos'),
    'E22': ('Número errado de ZIPs RF (esperado 27)',               'Re-execute 01_bootstrap_receita_federal.py'),
    'E23': ('ZIPs corrompidos',                                     'Delete ZIPs corrompidos, re-execute 01_bootstrap_receita_federal.py'),
    'E24': ('Estrutura antiga de compras_gov',                      'Delete compras_gov/, re-execute 04_bootstrap_compras.py'),
    # Seção 3 — Landing gate
    'E30': ('landing_gate.json ausente',                            'Execute 05_landing_validate.ipynb'),
    'E31': ('Gate status diferente de SUCCESS',                     'Corrija problemas (Seções 1-2), re-execute 05_landing_validate'),
    'E32': ('Gate desatualizado em relação ao bootstrap',           'Re-execute 05_landing_validate.ipynb após último bootstrap'),
    # Seção 4 — APIs
    'E40': ('Compras.gov erro HTTP ou JPA crash',                   'Aguarde 30-60 min (problema server-side)'),
    'E41': ('Compras.gov 400 tamanhoPagina',                        'Corrija PAGE_SIZE para 10-500 em 04_bootstrap_compras.py'),
    'E42': ('PNCP timeout ou 404',                                  'Tente novamente (P27); endpoint: /api/consulta/v1/contratos'),
    'E43': ('Transparencia 401/403 ou chave ausente',               'Renove chave em portaldatransparencia.gov.br'),
    'E44': ('Receita Federal 404/401',                              'Token rotacionado ou mês ainda nao publicado (dias 8-12)'),
    # Seção 5 — Bronze
    'E50': ('Diretório ou fonte Bronze ausente',                    'Execute o notebook Bronze correspondente'),
    'E51': ('Diretório Bronze sem arquivos Parquet',                'Re-execute o notebook Bronze correspondente'),
    'E52': ('Contagem Bronze abaixo do mínimo esperado',            'Re-execute o notebook Bronze correspondente'),
    'E53': ('Nao foi possivel ler Parquet Bronze',                  'Delete diretório, re-execute notebook Bronze'),
    'E54': ('bronze.duckdb ausente',                               'Normal antes de silver_identidade rodar'),
    'E55': ('bronze.duckdb.wal ou .tmp existem',                   'Delete se nenhum notebook estiver rodando — resíduo de conexão'),
    'E56': ('Tabelas RF ausentes em bronze.duckdb ou corrompido',   'Delete bronze.duckdb, re-execute 15_silver_identidade Step 2'),
    # Seção 6 — Silver
    'E60': ('Diretório ou fonte Silver ausente',                    'Execute notebooks Silver 13, 14, 15 em ordem'),
    'E61': ('Diretório Silver sem arquivos Parquet',                'Re-execute o notebook Silver correspondente'),
    'E62': ('Contagem Silver abaixo do mínimo esperado',            'Re-execute o notebook Silver correspondente'),
    'E63': ('Nao foi possivel ler Parquet Silver',                  'Delete diretório, re-execute notebook Silver'),
    'E64': ('Filtro ADR-007 nao aplicado — 70M+ rows em Silver',    'Delete silver_identidade/, re-execute Steps 3+4 na mesma sessão'),
    'E65': ('Falha de integridade cnpj_normalized',                 "Verifique REGEXP_REPLACE padrão '[^0-9A-Za-z]' — ADR-002"),
    # Seção 7 — Gold
    'E79': ('Arquivo Gold ausente',                                 'Execute o notebook Gold correspondente'),
    'E80': ('Contagem Gold fora do intervalo esperado',             'Re-execute o notebook Gold correspondente'),
    'E81': ('Nao foi possivel ler Parquet Gold',                    'Delete arquivo, re-execute notebook Gold'),
    'E82': ('SCD2 corrompido — duplicatas is_current=true',         'Delete dim_fornecedor.parquet, re-execute 17_dim_fornecedor.ipynb'),
    'E83': ('Grain de fato_contratos violado — contrato_sk duplicado', 'Inclua codigo_unidade_gestora no grain'),
    'E84': ('Percentual de valores negativos > 5% em fato_contratos', 'Verifique source Silver'),
    # Seção 8 — Serving
    'E85': ('Diretório ou arquivo Serving ausente',                 'Execute o notebook Serving correspondente'),
    'E86': ('Contagem Serving diferente de 810.921',                'Re-execute Step 15 do notebook 23'),
    'E87': ('Leakage ou CNPJ exposto em Serving',                   'Remova colunas de leakage — re-execute Step 15 do notebook 23'),
    'E88': ('Distribuição do target incorreta',                     'Re-execute Steps 3-15 do notebook 23'),
    'E89': ('Contagem de colunas diferente de 71',                  'Feature group ausente — re-execute Steps 4-15 do notebook 23'),
    'E90': ('CSV com quoting incorreto — vírgulas internas',        "Adicione QUOTE '\"' no COPY; use read_csv com quote='\"'"),
    # Seção 9 — DuckDB/utils
    'E91': ('ImportError: GOLD_DIR nao existe em utils.paths',      'Use gold_path(ROOT, table_name) — paths.py exporta funções'),
    'E92': ('BinderException: cnpj_token nao encontrado',           'Use supplier_sk — cnpj_token só existe em serving_perfil'),
    'E93': ('CSV quoting — vírgulas internas em campos categóricos', "COPY com QUOTE '\"'; read_csv com quote='\"'"),
    'E94': ('_valor_negativo nao existe em silver_pncp',            'Use valor_inicial > 0 para silver_pncp'),
    'E95': ('ModuleNotFoundError: sklearn',                         'pip install scikit-learn'),
    # Histórico
    'E70': ('try_strptime(BIGINT) Binder Error',                    'Use CASE WHEN val=0 THEN NULL ELSE TRY_STRPTIME(CAST(...))'),
    'E71': ('data.parquet duplicado no path',                       'Nao concatene /data.parquet — silver_path() já inclui'),
    'E72': ('Subtração datetime offset-naive/aware',                'Use datetime.now(timezone.utc) nos dois lados'),
    'E73': ('ModuleNotFoundError: paths',                           'Use resolução de 3 níveis com busca por marcador no Setup'),
    'E74': ('ADR-007 via partições antigas — 70M rows',             'Delete silver_identidade/, execute Steps 3+4 na mesma sessão'),
    'E75': ('PNCP campo esfera renomeado para esferaId',            'Substitua orgaoEntidade.esfera por orgaoEntidade.esferaId'),
    'E76': ('Row count FAIL após crescimento da fonte',             'Derive contagem esperada dinamicamente do Bronze com mesmo filtro'),
    'E77': ('FileNotFoundError: bronze.duckdb stat()',              'Proteja stat() com if DB_FILE.exists()'),
    'E78': ('SyntaxError: repr em condicional dentro de f-string',  'Extraia repr(val) para variável antes da f-string'),
}

print('=== Seção 10 — Lookup de códigos de erro ===')
print()
print(f"  {'Código':<6} {'Descrição':<55} Remediação")
print('  ' + '-' * 120)
prev_section = ''
for code_key, (desc, fix) in ERROR_CODES.items():
    section = code_key[1]
    if section != prev_section:
        print()
        prev_section = section
    print(f'  {code_key:<6} {desc:<55} {fix}')

print()

# Lookup interativo — defina o código e re-execute
lookup = ''
if lookup:
    if lookup in ERROR_CODES:
        desc, fix = ERROR_CODES[lookup]
        print(f'  [{lookup}] {desc}')
        print(f'  -> {fix}')
        if int(lookup[1:]) >= 70:
            print('  -> Veja Seção 9 para diagnóstico completo e padrão de correção')
    else:
        print(f'  Código {lookup!r} nao encontrado — verifique lista acima')

print()
print('  Seção 10 completa.')

---
## Apêndice — Problemas abertos conhecidos

| # | Problema | Status | Código |
|---|---|---|---|
| P27 | PNCP API timeout para dados históricos pré-2026 | Aberto | E42 |
| P28 | Compras.gov JPA crash durante re-download nacional | Aberto | E40 |
| P29 | Contratos pré-2021 (SICON) — sem API pública | Trabalho futuro | — |
| P31 | `privacy.py` corrigido (`Any` importado) pendente de commit | Alta | — |
| P26 | Portar notebooks Silver 13, 14, 15 para Databricks | Alta | — |

---
## Changelog

| Data | Mudança | Seções |
|---|---|---|
| 2026-04-28 | Versão inicial — cobertura até 05_landing_validate | 0–5 |
| 2026-05-07 | Estendido para Bronze (06–12) e Silver (13–15) | 0–8 |
| 2026-05-07 | Seção 5: verificações automatizadas Bronze (E50–E56) | 5 |
| 2026-05-07 | Seção 6: verificações automatizadas Silver (E60–E65) | 6 |
| 2026-05-07 | Seção 7: guia de referência 9 problemas DuckDB/utils (E70–E78) | 7 |
| 2026-05-07 | Setup: resolução de root funciona de qualquer profundidade | Setup |
| 2026-05-08 | Seção 7: Gold layer com checks SCD2 e grain (E79–E84) | 7 |
| 2026-05-08 | Seção 8: Serving layer com leakage/LGPD/CSV checks (E85–E90) | 8 |
| 2026-05-08 | Seção 9: 5 novos erros da sessão 08/05 (E91–E95) | 9 |
| 2026-05-08 | Setup: gold_path, serving_path, silver_path adicionados | Setup |
| 2026-05-08 | Seção 0: sklearn adicionado aos pacotes obrigatórios | 0 |